# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amin119/FlyRank-AI-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My lane: Lane 2 — Refresh / Content Opportunity Scoring** (not freestyle).

FlyRank's clients don't have a "too little data" problem — they have too much of it and not enough reviewer hours. The starter dataset alone holds 30,000 pages; a real client portfolio is bigger still, and nobody can eyeball a spreadsheet that size every week. This lane produces exactly the artifact a stretched content team needs: a short, ranked, explainable list of pages to look at first, each with a reason attached. I picked it over the other three lanes because:

- Lane 1 (signal analysis) and Lane 3 (archetype clustering) are diagnostic — they describe the landscape but don't hand anyone a to-do list.
- Lane 4 (CTR/engagement scoring) is a narrower version of the same "which page first" decision; Lane 2 already folds CTR and engagement reason codes in as inputs alongside staleness and demand, so it's the more general framing.
- Lane 2 is the one lane where the starter repo already ships a working end-to-end pipeline (`scripts/01`–`05`) I can inspect, question, and try to beat, rather than starting from a blank page. Its own committed results (see Section 3) already show a fixed rule and a learned model give very different answers on the *same* data — that's the first real evidence this direction is worth 7 weeks, not just a repo I happened to be handed.

This is a provisional pick. I can confirm or switch lanes any time up to the end of Week 4, once the signal audit and baseline work (Weeks 2–4) tell me more about whether the signal actually holds up.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The question:** given a content team with limited review capacity, which existing pages should be reviewed first this week — and for what reason?

**Unit of analysis:** one *page* (one pseudonymized content item). In the starter dataset that's one row of `content_refresh_anonymized.csv` — a trailing-90-day snapshot of a single piece of content for one client. Not a client, not a day: the action ("refresh this page's copy") is taken on a page, so the page is the thing being scored and ranked.

**Output:** a ranked queue — page id, a numeric opportunity/risk score, a suggested action (`refresh`, `expand_and_refresh`, `refresh_and_review_ctr`, `refresh_and_review_engagement`, `protect`/`monitor`), and one or more reason codes a human can check in ten seconds (e.g. `stale_visible_page`, `declining_with_demand`).

**Who acts, and how:** a content editor or SEO reviewer with a fixed weekly capacity (say, 20–50 pages they can realistically look at). They open the top of the queue, read the reason codes, and decide: rewrite or update the page, expand a thin section, fix a title/meta for CTR, or leave it alone (`protect`) because it's actually fine.

**Cost of a wrong call — and it's asymmetric:**
- *False positive* (queue says "review me," page was actually healthy): costs a reviewer's scarce time on a page that didn't need it. Do that often enough across a week's queue and the real cost isn't the wasted hour — it's the team learning to distrust the queue.
- *False negative* (a genuinely declining, high-demand page never surfaces): the page keeps losing visibility and clicks silently. Decline tends to compound the longer it goes unnoticed, so this miss gets more expensive over time, not less.
- Because reviewer capacity is the scarce resource, what matters is not "is the model accurate on average" but "are the top 20–50 items in the queue actually worth reviewing" — precision@K is the metric that matches the real decision, not plain accuracy.

**Why data/ML helps at all:** a page can be worth reviewing for several independent reasons at once — stale *and* still in demand, or well-positioned but under-clicked, or declining alongside its whole topic (not its own fault). No single hand-written if-statement can weigh staleness, demand, position, CTR gap, and trend persistence against each other for 30,000 pages without either missing real opportunities or drowning reviewers in false alarms. That is a ranking problem across many correlated, messy signals — the kind of problem where a learned model can earn its place over a fixed rule, *if* it can be shown to actually beat that rule honestly.

Notice what is deliberately not in this paragraph yet: a model. The real work is defining the decision, the label, the validation split (client holdout, so I'm not scored on pages whose client I've effectively already seen), the baseline to beat, and the action mapping. A model is one component inside that pipeline — worthless without this framing, and untrustworthy without honest validation. "Train a model" is the easy 5% once the other 95% is right.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import os, sys, subprocess, re

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/amin119/FlyRank-AI-internship"
REPO_DIR = "FlyRank-AI-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks" and os.path.basename(os.path.dirname(os.getcwd())) == "work":
    os.chdir("../..")  # move from work/notebooks/ to the repo root

print("Working dir:", os.getcwd())

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Starter dataset: {df.shape[0]:,} pages x {df.shape[1]} columns, {df['client_id'].nunique()} clients\n")

# Number 1 — how many pages the CURRENT simple rule already flags as "declining"
declining = df["trend_direction"] == "down"
print(f"1) {declining.sum():,} of {len(df):,} pages ({declining.mean()*100:.1f}%) are flagged 'declining' "
      f"by the simple trend rule alone.")
print("   -> Over half the inventory 'looks bad' by one rule. No reviewer reads 16k pages one by one;")
print("      this is a ranking problem, not a flagging problem.\n")

# Number 2 — of those, how many still carry real search demand (so a wasted review actually matters)
declining_with_demand = df[declining & (df["impressions_90d"] >= 100)]
print(f"2) {len(declining_with_demand):,} pages ({len(declining_with_demand) / len(df) * 100:.1f}% of all pages) "
      f"are BOTH declining and still pulling >=100 impressions/90d.")
print("   -> 'Declining' isn't a rare edge case here - it's a large, real-traffic population,")
print("      exactly the kind of pattern worth ranking well instead of eyeballing.\n")

# Number 3 — does a learned ranking actually beat the fixed rule on THIS data?
# (the starter pipeline's own committed report, generated by running scripts/01-04 on this same CSV)
with open("outputs/model_report.md", encoding="utf-8") as f:
    report = f.read()

precision_at_50 = dict(re.findall(r"\|\s*(\w+)\s*\|\s*[\d.]+\s*\|\s*[\d.]+\s*\|\s*([\d.]+)\s*\|", report))
baseline_p50 = float(precision_at_50["baseline_rules"])
rf_p50 = float(precision_at_50["random_forest"])
print("3) In the starter pipeline's own committed results (outputs/model_report.md), of the top 50 pages "
      "each method ranks first:")
print(f"   - the fixed baseline rule gets {baseline_p50 * 50:.0f}/50 right (precision@50 = {baseline_p50})")
print(f"   - a random forest gets {rf_p50 * 50:.0f}/50 right (precision@50 = {rf_p50})")
print(f"   -> roughly {rf_p50 / baseline_p50:.1f}x more true positives in the exact list length a reviewer")
print("      would actually work through — on a 30k-row starter slice, under client-holdout validation.")
print("      That gap is the strongest early evidence this lane is worth 7 weeks: a smarter ranking")
print("      measurably beats a rule anyone could already write today.")

Working dir: C:\Users\METIS-005\Favorites\Dell\Fkyrank AI\FlyRank-AI-internship


Starter dataset: 30,000 pages x 44 columns, 32 clients

1) 16,262 of 30,000 pages (54.2%) are flagged 'declining' by the simple trend rule alone.
   -> Over half the inventory 'looks bad' by one rule. No reviewer reads 16k pages one by one;
      this is a ranking problem, not a flagging problem.

2) 13,152 pages (43.8% of all pages) are BOTH declining and still pulling >=100 impressions/90d.
   -> 'Declining' isn't a rare edge case here - it's a large, real-traffic population,
      exactly the kind of pattern worth ranking well instead of eyeballing.

3) In the starter pipeline's own committed results (outputs/model_report.md), of the top 50 pages each method ranks first:
   - the fixed baseline rule gets 12/50 right (precision@50 = 0.24)
   - a random forest gets 37/50 right (precision@50 = 0.74)
   -> roughly 3.1x more true positives in the exact list length a reviewer
      would actually work through — on a 30k-row starter slice, under client-holdout validation.
      That gap is

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work will be able to say:**
- *Observed*: "In this 30,000-page anonymized starter slice, pages with a given combination of signals were more or less likely to carry the current-window 'declining' label." An observed association, not a law.
- *Directional*: "Stale, high-demand, low-CTR pages look more likely to need attention than fresh, low-demand pages." A direction, not a guarantee for any one specific page.
- *Decision-support*: "This queue ranks pages by estimated review priority under limited reviewer capacity." A tool that orders candidates for a human to check, not a verdict.

**What this work will never say:**
- **Not causal.** I cannot claim that refreshing a page *causes* recovery. Section 3's comparison is entirely about ranking quality (precision@K), never about proving an intervention works — that needs an actual experiment (e.g. before/after on refreshed vs. matched unrefreshed pages), which is out of scope here.
- **Not "predicting Google."** Nothing here predicts or reverse-engineers a search engine's ranking algorithm. Every signal used is an observed outcome — impressions, clicks, position, engagement — never the algorithm's internals.
- **Not a benchmark on the full warehouse.** Every number in Section 3 comes from a 30,000-row anonymized *starter* slice, validated with a client holdout. It's evidence this lane is worth pursuing — not a result I can quote once I move to the ~79M-row warehouse in Weeks 3–5, where it has to be re-earned with its own leakage checks.
- **Not blind to the label's own weakness.** `is_declining_label` (and `trend_direction`) is a *current-window* proxy, not a true future outcome — it's computed from the same 90-day window the features come from. Before I lean on any model score, I need to either justify that proxy carefully or replace it with a real forward-looking label (prior 90 days → next 30 days), which the lane guide flags as the stronger target for this exact dataset.
- **No client names, domains, URLs, or raw queries** appear anywhere in this notebook, and none will appear in later ones — the starter data ships pseudonymized and safe by design, and I'm keeping it that way.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.